# pipeline_silver

Parametrized Silver cleansing notebook — Bronze (CDF) → quality routing → Delta MERGE.

One notebook, 12 Silver domains. DABs passes domain-specific widgets per task.
See ADR-002 (Bronze/Silver lineage) and ADR-004 (Liquid Clustering = merge_key).

In [ ]:
# Widget defaults match payment_events; DABs overrides per domain task.
dbutils.widgets.removeAll()

dbutils.widgets.text("table_name",       "payment_events")
dbutils.widgets.text("bronze_table",     "ubereats_dev.bronze.payment_events")
dbutils.widgets.text("silver_table",     "ubereats_dev.silver.payment_events")
dbutils.widgets.text("quarantine_table", "ubereats_dev.quarantine.payment_events")
dbutils.widgets.text("contract_path",    "contracts/payment_events.yml")
dbutils.widgets.text("checkpoint_path",  "/Volumes/ubereats_dev/checkpoints/silver/payment_events")

In [ ]:
table_name      = dbutils.widgets.get("table_name")
bronze_table    = dbutils.widgets.get("bronze_table")
silver_table    = dbutils.widgets.get("silver_table")
quarantine_table = dbutils.widgets.get("quarantine_table")
contract_path   = dbutils.widgets.get("contract_path")
checkpoint_path = dbutils.widgets.get("checkpoint_path")

print(f"[silver] table={table_name}  bronze={bronze_table}  silver={silver_table}")

In [ ]:
import sys

# contracts/ package lives at project root, one level above notebooks/
sys.path.insert(0, "..")

from contracts.loader import load_contract
from contracts.spark_schema import to_create_table_ddl

contract  = load_contract(contract_path)
merge_key = contract["table"]["merge_key"]

# Silver and Quarantine share the same schema (quarantine mirrors Silver — CLAUDE.md)
silver_ddl     = to_create_table_ddl(contract, silver_table)
quarantine_ddl = to_create_table_ddl(contract, quarantine_table)

spark.sql(silver_ddl)
spark.sql(quarantine_ddl)

print(f"[silver] tables ready  merge_key={merge_key}")
print(silver_ddl)

In [ ]:
from pyspark.sql.functions import col, current_timestamp, lit


def _rule_fail_expr(rule):
    """Returns a boolean Column: True when the record FAILS the rule."""
    field = rule["field"]
    check = rule["check"]
    if check == "not_null":
        return col(field).isNull()
    if check == "allowed_values":
        # Only flag if field is non-null and not in the allowed set
        return col(field).isNotNull() & ~col(field).isin(rule["values"])
    if check == "not_future":
        return col(field).isNotNull() & (col(field).cast("timestamp") > current_timestamp())
    return lit(False)


def apply_quality_rules(df, contract):
    """
    Splits df into (clean_df, quarantine_df) based on Silver quality rules.

    - on_failure=quarantine → routes record to quarantine_df
    - on_failure=warn       → record stays in clean_df (warning logged by caller)
    """
    quarantine_rules = [
        r for r in contract["quality"]["rules"]
        if "silver" in r["scope"] and r["on_failure"] == "quarantine"
    ]
    if not quarantine_rules:
        return df, df.filter(lit(False))

    fail_expr = _rule_fail_expr(quarantine_rules[0])
    for r in quarantine_rules[1:]:
        fail_expr = fail_expr | _rule_fail_expr(r)

    return df.filter(~fail_expr), df.filter(fail_expr)

In [ ]:
# CDF columns added by Delta — dropped before quality checks and MERGE
_CDF_COLS = {"_change_type", "_commit_version", "_commit_timestamp"}

# Bronze tables have delta.enableChangeDataFeed=true (set via contract TBLPROPERTIES)
# Bronze is immutable (INSERT-only), so _change_type will always be 'insert'
bronze_stream = (
    spark.readStream
    .format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", "0")
    .table(bronze_table)
    .filter(col("_change_type") == "insert")
    .drop(*_CDF_COLS)
)

In [ ]:
def process_silver_batch(batch_df, batch_id):
    if batch_df.isEmpty():
        return

    clean_df, quarantine_df = apply_quality_rules(batch_df, contract)

    if not quarantine_df.isEmpty():
        (quarantine_df
         .write
         .format("delta")
         .mode("append")
         .saveAsTable(quarantine_table))
        print(f"[silver] batch {batch_id}: {quarantine_df.count()} records → quarantine")

    if clean_df.isEmpty():
        return

    view = f"silver_batch_{table_name}"
    clean_df.createOrReplaceTempView(view)

    # ADR-04: MERGE ON merge_key aligns with Liquid Clustering → file pruning
    # UPDATE only when incoming record has a newer source timestamp
    spark.sql(f"""
        MERGE INTO {silver_table} AS t
        USING {view} AS s
        ON t.`{merge_key}` = s.`{merge_key}`
        WHEN MATCHED AND s.__source_ts_ms > t.__source_ts_ms THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)


(
    bronze_stream
    .writeStream
    .foreachBatch(process_silver_batch)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
    .awaitTermination()
)